# Zipf's and Heaps' Laws: The Statistics of Vocabulary

Two regularities shape every corpus. **Zipf's law**: the $r$-th most frequent word type occurs with
frequency roughly proportional to $r^{-s}$. **Heaps' law**: after $n$ tokens we have seen roughly
$n^{\beta}$ distinct types. This notebook walks the topic's argument with every number computed by
`zipf_heaps_vocabulary_statistics.py`, which owns them; nothing here redefines the math.

The arc: rank–frequency is a tail distribution turned sideways (Proposition 1); Zipf is the
maximum-entropy law at fixed mean log-rank; Zipf implies Heaps with $\beta = 1/s$ (Karlin's theorem);
the Pitman–Yor process plants both laws through one parameter; and estimating the exponent is harder
than drawing a line through a log–log plot. The finance case study closes the loop on a corpus where
we *know* the answer.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import numpy as np
from scipy import special

here = Path.cwd()
sys.path.insert(0, str(here if (here / "zipf_heaps_vocabulary_statistics.py").exists()
                       else here / "notebooks" / "zipf-heaps-vocabulary-statistics"))
import zipf_heaps_vocabulary_statistics as zh

fc = zh.fc  # the shared finance corpus module
st = zh.finance_streams()
counts = zh.rank_frequency(list(st["full"]))
print(f"{len(st['full'])} tokens, {len(counts)} types, {np.mean(counts == 1):.1%} of types seen once")
print("top ten types:", [w for w, _ in Counter(st["full"]).most_common(10)])

## 1. The shape of a vocabulary

Sort the type counts in decreasing order, $f(1) \ge f(2) \ge \dots \ge f(V)$, and plot $\log f(r)$
against $\log r$. A power law $f(r) \propto r^{-s}$ is a straight line of slope $-s$. Here are a few
log-spaced points from the finance corpus; the head is the closed class of template words, the tail
is the open class of content words.

In [ ]:
for r in zh.log_spaced_ranks(len(counts), 12):
    print(f"rank {r:5d}   count {int(counts[r - 1]):5d}")

## 2. Rank–frequency is the count tail turned sideways

**Proposition 1.** With ties ranked last, the rank of a type seen $f$ times is
$r(f) = \#\{\text{types with count} \ge f\}$. So the rank–frequency curve is the (unnormalized)
complementary CDF of the type counts with its axes swapped. If the count spectrum has a power tail,
$\#\{\text{count} \ge f\} \propto f^{-(\alpha - 1)}$, then $r \propto f^{-(\alpha-1)}$, i.e.
$f \propto r^{-1/(\alpha-1)}$. **Corollary 1:** the exponents are tied by $s = 1/(\alpha - 1)$, $\alpha = 1 + 1/s$.

The harness checks the identity exactly on the corpus, and the exponent map on random power-law
samples.

In [ ]:
spectrum = zh.frequency_spectrum(counts)
print("V_1, V_2, V_3 =", spectrum[1], spectrum[2], spectrum[3])
for f in (1, 2, 10, 100):
    print(f"r({f}) = #{{count >= {f}}} = {zh.rank_of_frequency(counts, f)}")
zh.test_rank_is_count_tail()
zh.test_spectrum_exponent_maps_to_rank_exponent()
print("alpha = 1.6  <=>  s =", round(zh.zipf_exponent_from_alpha(1.6), 4))

## 3. Zipf as a probability law

On an infinite vocabulary, $p_r = r^{-s}/\zeta(s)$ needs $\zeta(s) = \sum_{r \ge 1} r^{-s} < \infty$,
which holds exactly when $s > 1$; at $s = 1$ the partial sums grow like $\log R + \gamma$.

**Why this shape? (Mandelbrot.)** Among all distributions on $\{1, \dots, R\}$ with a fixed mean
log-rank $\mathbb{E}[\log r] = \mu$, entropy is maximized by $p_r \propto r^{-s}$, where $s$ is the
Lagrange multiplier that meets the constraint. The harness solves the constrained problem with a
generic optimizer and recovers the Zipf law.

In [ ]:
zh.test_zeta_converges_iff_s_above_one()
R = 40
p = zh.zipf_pmf(1.3, R)
mu = float(p @ np.log(np.arange(1, R + 1)))
print(f"mu = {mu:.4f} -> Lagrange multiplier s = {zh.maxent_exponent(mu, R):.6f}")
zh.test_zipf_maximizes_entropy_at_fixed_log_rank()
print("SLSQP max-entropy solution matches Zipf(1.3) to 1e-4")

## 4. Zipf implies Heaps (Karlin 1967)

Draw $n$ tokens i.i.d. from $p_r = C r^{-s}$, $C = 1/\zeta(s)$. Type $r$ is seen with probability
$1 - (1 - p_r)^n$, so $\mathbb{E}\,V(n) = \sum_r [1 - (1 - p_r)^n]$.

**Theorem 2.** For $s > 1$, $\mathbb{E}\,V(n) \sim \Gamma(1 - 1/s)\,(Cn)^{1/s}$, so Heaps' law holds
with $\beta = 1/s$.

The proof has two steps, both asserted below. (i) Replace $(1-p)^n$ by $e^{-np}$: the summand
$g(x) = 1 - e^{-nCx^{-s}}$ decreases in $x$, so the sum is sandwiched between $I - 1$ and $I$, where
$I = \int_0^\infty g = \Gamma(1 - 1/s)(Cn)^{1/s}$. (ii) Since $(1-p)^n \le e^{-np}$, the true sum is
larger, but only by a lower-order amount.

In [ ]:
for s in zh.KARLIN_S:
    row = [zh.expected_vocab(s, n) / zh.karlin_integral(s, n) for n in zh.KARLIN_N]
    print(f"s = {s:<5} E V(n) / Gamma(1-1/s)(Cn)^(1/s):", "  ".join(f"{x:.4f}" for x in row))
zh.test_karlin_poissonized_sandwich()
zh.test_karlin_depoissonization_gap_is_lower_order()
zh.test_karlin_zipf_implies_heaps()

Two consequences worth seeing. **Convergence is slowest for large $s$**: the sandwich error is at most
one type, against $I \sim n^{1/s}$ types. And **Heaps needs an open vocabulary**: truncate Zipf at $R$
types and $V(n) \le R$, so the growth curve flattens.

**Corollary (hapax share).** $\mathbb{E}\,V_1(n) / \mathbb{E}\,V(n) \to 1/s$, where $V_1$ counts
types seen exactly once — the share of hapax legomena estimates the Heaps exponent.

In [ ]:
p = zh.zipf_pmf(1.25, 10_000)
for n in (10**3, 10**4, 10**6, 10**7):
    print(f"truncated Zipf (R = 10,000, s = 1.25): E V({n:>8}) = {zh.expected_vocab_finite(p, n):8.1f}")
zh.test_finite_vocabulary_saturates()
for s in (1.5, 2.0, 3.0):
    share = zh.expected_hapax(s, 10**6) / zh.expected_vocab(s, 10**6)
    print(f"s = {s}: hapax share at n = 1e6 is {share:.4f}  (1/s = {1 / s:.4f})")
zh.test_karlin_hapax_share()

## 5. Pitman–Yor plants both laws

The shared corpus generates content words from a Pitman–Yor Chinese restaurant $\mathrm{PY}(d, \theta)$:
customer $n+1$ opens a new table (a new word type) with probability $(\theta + dK_n)/(\theta + n)$, or
joins table $k$ with probability $(c_k - d)/(\theta + n)$.

**Theorem 3.** $\mathbb{E}[K_n] + \theta/d = \dfrac{\theta}{d}\,
\dfrac{\Gamma(\theta + n + d)\,\Gamma(\theta)}{\Gamma(\theta + d)\,\Gamma(\theta + n)}
\sim \dfrac{\Gamma(\theta + 1)}{d\,\Gamma(\theta + d)}\, n^{d}.$

Proof: taking expectations of the seating rule gives
$\mathbb{E}[K_{n+1}] = \mathbb{E}[K_n] + (\theta + d\,\mathbb{E}[K_n])/(\theta + n)$, so
$M_n = \mathbb{E}[K_n] + \theta/d$ satisfies $M_{n+1} = M_n\,(\theta + n + d)/(\theta + n)$ with
$M_0 = \theta/d$; telescope. Note the offset: it is $\mathbb{E}[K_n] + \theta/d$, not $\mathbb{E}[K_n]$,
that grows as a clean power, in $n + \theta$ rather than $n$ (Wendel's inequality bounds its ratio to
the constant above times $(n + \theta)^d$ between $((\theta + n)/(\theta + n + d))^{1-d}$ and $1$). As a power of $n$ it is clean only asymptotically.

In [ ]:
zh.test_py_exact_formula_solves_recursion()
zh.test_py_monte_carlo_matches_exact_mean()
zh.test_py_matches_corpus_sampler()
zh.test_py_growth_is_n_to_the_d()
d, theta = fc.HEAPS_DISCOUNT, fc.PY_STRENGTH
for n in (10**2, 10**3, 10**4, 10**6):
    ek = float(zh.py_expected_types(n, d, theta))
    print(f"n = {n:>8}: E[K_n] = {ek:10.1f}   E[K_n]/(c n^d) = {ek / float(zh.py_asymptote(n, d, theta)):.4f}")

The count spectrum of a Pitman–Yor restaurant converges to the **Sibuya** law
$K_{n,m}/K_n \to d\,\Gamma(m - d)/(\Gamma(1-d)\, m!)$ (Pitman 2006, Lemma 3.11), whose tail decays as
$m^{-(1+d)}$. By Corollary 1, $\alpha = 1 + d$ means the rank–frequency exponent is $s = 1/d$: the
discount sets Heaps' $\beta = d$ and Zipf's $s = 1/d$ at once, exactly as Theorem 2 requires. In
particular the hapax share tends to $d$.

In [ ]:
zh.test_py_spectrum_is_sibuya()
m = np.arange(1, 6)
for dd in (0.4, 0.6, 0.8):
    print(f"d = {dd}: Sibuya shares m=1..5:", np.round(zh.sibuya_pmf(m, dd), 4))

## 6. Estimating the exponent

The classical recipe — fit a line to the log–log plot by ordinary least squares — is fragile. On the
finance corpus, the dense OLS slope depends heavily on which ranks we fit:

In [ ]:
for (lo, hi), slope in zip(zh.LAB_WINDOWS, [zh.zipf_slope_window(counts, lo, hi) for lo, hi in zh.LAB_WINDOWS]):
    print(f"ranks {lo:>4}-{hi:<5} OLS slope {slope:+.3f}")
zh.test_ols_slope_depends_on_window()

The alternative (Clauset, Shalizi, and Newman 2009) treats each type's count $x_i \ge x_{\min}$ as a
draw from the discrete power law $P(X = x) = x^{-\alpha}/\zeta(\alpha, x_{\min})$ with the Hurwitz zeta
normalizer, and maximizes
$\ell(\alpha) = -n \log \zeta(\alpha, x_{\min}) - \alpha \sum_i \log x_i$. The score equation is
$\frac{1}{n}\sum_i \log x_i = -\partial_\alpha \log \zeta(\alpha, x_{\min})$; $x_{\min}$ is chosen to
minimize the Kolmogorov–Smirnov distance. On random power-law samples the MLE is consistent and
matches `scipy.stats.fit`; OLS on the frequency histogram is badly biased, and OLS on the rank plot
is at least twice as far off as the MLE.

In [ ]:
zh.test_mle_is_consistent_and_matches_scipy()
zh.test_mle_score_equation()
zh.test_continuous_approximation_close_for_large_xmin()
zh.test_ols_is_biased_where_mle_is_not()
xmin, alpha = zh.select_xmin(counts)
print(f"full text: KS-selected xmin = {xmin}, alpha = {alpha:.4f}, s = 1/(alpha-1) = {zh.zipf_exponent_from_alpha(alpha):.4f}")

## 7. Finance case study: a corpus with a planted answer

`finance_corpus.py` draws its content words from Pitman–Yor restaurants with discount
`HEAPS_DISCOUNT` $= 0.6$ and strength `PY_STRENGTH` $= 20$ (one shared restaurant plus one per sector),
and wraps them in closed-class templates. So we know the truth: $\beta = d = 0.6$ and $s = 1/d \approx 1.667$.
We rebuild the corpus while recording every restaurant draw (the rebuild is byte-identical to
`generate_corpus(300, 0)`).

**Full text looks like real text.** The log-spaced OLS gives $s \approx 0.92$ and Heaps gives
$\beta \approx 0.52$ — the familiar real-text pairing that violates $\beta = 1/s$ by about a factor of
two, although one parameter generated everything. A closed-class head on an open-class tail produces
that tension by itself.

In [ ]:
S = zh.finance_summary()
print(f"full text:  OLS Zipf s = {-S['zipf_slope_logspaced']:.3f}, Heaps beta = {S['heaps_full']:.3f}, "
      f"1/s = {1 / -S['zipf_slope_logspaced']:.3f}")
zh.test_finance_rebuild_matches_corpus()
zh.test_ols_matches_corpus_helper()
zh.test_full_text_shows_the_real_text_tension()
zh.test_templates_saturate_full_text_growth()

**Generic estimators miss the planted truth.** Even on the pure content stream, the naive Heaps slope
overshoots $d$ (it ignores Theorem 3's $-\theta/d$ offset), and the power-law MLE's $\hat\alpha$ drifts
with $x_{\min}$, far above $1 + d$: the Sibuya spectrum is a power law only for counts $\gg 1$, which a
12k-draw stream barely has.

**Theorem 3 recovers it.** On the largest restaurant (the shared one), fitting
$\log(K_n + \theta/d)$ against $\log(n + \theta)$ — the theorem's exact form, used as an estimator —
lands on $d$.

In [ ]:
print(f"content stream: naive Heaps = {S['heaps_content_naive']:.3f}, hapax share = {S['hapax_share_content']:.3f}, "
      f"power-law alpha over xmin in {zh.POWERLAW_XMINS}: {S['alpha_content_min']:.3f} .. {S['alpha_content_max']:.3f}")
print(f"shared restaurant (n = {S['N_common']}): naive Heaps = {S['heaps_common_naive']:.3f}, "
      f"offset-corrected = {S['heaps_common_shifted']:.4f}  (planted d = {fc.HEAPS_DISCOUNT})")
zh.test_generic_estimators_miss_the_planted_truth()
zh.test_theorem_three_recovers_the_planted_discount()

## 8. The lab's numbers, and the full harness

`viz_payload()` holds every number the interactive lab displays. The lab's second panel claims that
the offset-corrected Heaps slope tracks the discount slider while the naive slope overshoots.

In [ ]:
P = zh.viz_payload()
print("OLS slope by window:", P["windows"])
print("MLE:", P["mle"])
for dd, g in P["growth"].items():
    print(f"d = {dd}: K_20000 = {g['K'][-1]:5d}, E[K] = {g['EK'][-1]:8.1f}, "
          f"naive Heaps = {g['heaps_naive']:.3f}, corrected = {g['heaps_shifted']:.3f}")

In [ ]:
assert zh._run_harness() == 0